In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
np.random.seed(109204)
plt.rcParams.update({'axes.labelsize': 18, 'xtick.labelsize': 16, 'ytick.labelsize': 16, 'axes.titlesize': 18,})

def logistic_map(x, r): # Function for Logistic
    return r * x * (1 - x)

sigma_Q = 0.5  # Controls Variability of Q_k
def ukf_logistic_r_estimation(r_true, r_guess, n_steps=100, Q=1e-5, Q_seq = 0.5, x0_true=0.7, x0_hat=0.7, y_meas=None, alpha=1e-3, beta=2, kappa=0):
    n = 2 
    lam = alpha**2 * (n + kappa) - n # Lambda substitution
    c = n + lam # Denominator of w
    Wm = np.full(2*n + 1, 1/(2*c)) # For Mean
    Wc = np.full(2*n + 1, 1/(2*c)) # For Variance
    Wm[0] = lam/c
    Wc[0] = (-alpha**2 + beta) + lam/c

    generated = False
    Q0 = Q
    Q_seq = Q0 * np.exp(sigma_Q * np.random.randn(n_steps))
    if y_meas is None:
        generated = True
        x_true = np.zeros(n_steps)
        y_true = np.zeros(n_steps)
        x_true[0] = x0_true
        y_meas[0] = x_true[0] + np.random.normal(0, np.sqrt(Q_seq[0]))
        for k in range(1, n_steps):
            x_true[k] = logistic_map(x_true[k-1], r_true)
            y_meas[k] = x_true[k] + np.random.normal(0, np.sqrt(Q_seq[k]))
    else:
        x_true = None # If data does already exist, just use that

    x_hat = np.zeros(n_steps)
    r_hat = np.zeros(n_steps)
    x_hat[0] = x0_hat
    r_hat[0] = r_guess
    P = np.eye(n)

    def sigma_points(mu, P): # Sigma Point Generator
        U, s, _ = np.linalg.svd(c * P, full_matrices=False)
        s = np.maximum(s, 1e-12) # SVD
        A = U @ np.diag(np.sqrt(s))

        SP = np.zeros((2*n + 1, n))
        SP[0] = mu
        
        for i in range(n):
            SP[i + 1] = mu + A[:, i]
            SP[n + i + 1] = mu - A[:, i]
        return SP

    for k in range(1, n_steps): # UKF Loop
        mu = np.array([x_hat[k-1], r_hat[k-1]]) # Prediction Stage
        X = sigma_points(mu, P) # Form Sigma Points
        X_pred = np.zeros_like(X) # Propagate SP through Phi
        for i in range(2 * n + 1):
            x_i, r_i = X[i]
            x_next = logistic_map(x_i, r_i)
            r_next = r_i  # Assume constant r
            X_pred[i] = np.array([x_next, r_next])

        mu_pred = np.sum(Wm[:, None] * X_pred, axis=0) # Predict Mean

        P_pred = np.zeros((n, n)) # Predict Covariance
        for i in range(2 * n + 1):
            d = (X_pred[i] - mu_pred).reshape(-1, 1)
            P_pred += Wc[i] * (d @ d.T)

        Y_sigma = np.zeros(2*n + 1) # Update Stage
        for i in range(2*n + 1):
            Y_sigma[i] = X_pred[i, 0]

        y_pred = np.sum(Wm * Y_sigma) # Measurement for Mean

        S = 0 # Measurement for Variance Covariance
        for i in range(2*n + 1):
            dy = Y_sigma[i] - y_pred
            S += Wc[i] * (dy*dy)
        S += Q_seq[k]

        P_xy = np.zeros((n, 1)) # Cross-Covariance between State & Measurement
        for i in range(2 * n + 1):
            dx = (X_pred[i] - mu_pred).reshape(-1, 1)
            dy = (Y_sigma[i] - y_pred)
            P_xy += Wc[i] * (dx*dy)

        K = P_xy/S
        innovation = y_meas[k] - y_pred
        mu_upd = mu_pred + (K.flatten() * innovation) # Update Mean & Covariance
        P_upd = P_pred - (K @ K.T) * S
        x_hat[k] = mu_upd[0]
        r_hat[k] = mu_upd[1]
        P = P_upd
    return x_true, y_meas, x_hat, r_hat

def interactive_ukf(r_true=2.8, Q=1e-5, n_steps=100, alpha=1e-3, beta=2, kappa=0):
    r_guess = 1
    n_runs = 100

    Q0 = Q
    Q_seq = Q0 * np.exp(sigma_Q * np.random.randn(n_steps)) 
    x_true_base = np.zeros(n_steps) # True Logistic Map (Single Trajectory)
    y_meas_base = np.zeros(n_steps)
    x_true_base[0] = 0.7
    y_meas_base[0] = x_true_base[0] + np.random.normal(0, np.sqrt(Q_seq[0]))
    for k in range(1, n_steps):
        x_true_base[k] = logistic_map(x_true_base[k-1], r_true)
        y_meas_base[k] = x_true_base[k] + np.random.normal(0, np.sqrt(Q_seq[k]))

    all_x_true = np.zeros((n_runs, n_steps))
    all_x_hat = np.zeros((n_runs, n_steps))
    all_r_hat = np.zeros((n_runs, n_steps))

    for i in range(n_runs):
        y_meas = x_true_base + np.random.normal(0, np.sqrt(Q), size=n_steps)
        x_true, y_used, x_hat, r_hat = ukf_logistic_r_estimation(
            r_true=r_true, r_guess=r_guess, n_steps=n_steps, Q=Q,
            x0_true=0.7, x0_hat=0.7, y_meas=y_meas )
        all_x_true[i, :] = x_true_base
        all_x_hat[i, :] = x_hat
        all_r_hat[i, :] = r_hat
    x_hat_mean = np.mean(all_x_hat, axis=0)
    x_hat_p05 = np.percentile(all_x_hat, 5, axis=0)
    x_hat_p95 = np.percentile(all_x_hat, 95, axis=0)
    mse_t = np.mean((all_x_hat - all_x_true) ** 2, axis=0)
    variance_t = np.var(all_x_hat, axis=0)

    fig, ax = plt.subplots(2, 2, figsize=(12, 6), sharex=True)

    ax[0, 0].plot(x_true_base, 'k-', lw=2, label='True Logistic Map')
    ax[0, 0].plot(x_hat_mean, 'b--', lw=1.5, label='Mean UKF Estimate')
    ax[0, 0].plot(y_meas_base, 'r.', alpha=0.4, label='Measurements')
    ax[0, 0].fill_between(range(n_steps), x_hat_p05, x_hat_p95, alpha=0.2, label='95% Confidence Band')
    ax[0, 0].set_title(f'UKF Monte Carlo (r_true={r_true:.3f}, Q={Q:.1e})')
    ax[0, 0].set_ylabel('x')
    ax[0, 0].legend()

    r_hat_mean = np.mean(all_r_hat, axis=0)
    r_hat_p05 = np.percentile(all_r_hat, 5, axis=0)
    r_hat_p95 = np.percentile(all_r_hat, 95, axis=0)
    ax[0, 1].axhline(r_true, color='k', lw=2, label='True r')
    ax[0, 1].plot(r_hat_mean, 'b--', lw=1.5, label='Mean UKF Estimate')
    ax[0, 1].fill_between(range(n_steps), r_hat_p05, r_hat_p95, alpha=0.2, label='95% Confidence Band')
    ax[0, 1].set_title('UKF Parameter Evolution Across MC Runs')
    ax[0, 1].set_ylabel('r')
    ax[0, 1].legend()

    ax[1, 0].plot(mse_t, 'b-', label='MSE')
    ax[1, 0].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[1, 0].set_xlabel('Iteration')
    ax[1, 0].set_ylabel('MSE')
    ax[1, 0].set_title('MSE Evolution Over Time')
    ax[1, 0].legend()

    ax[1, 1].plot(variance_t, 'r-', label='MC Variance')
    ax[1, 1].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[1, 1].set_xlabel('Iteration')
    ax[1, 1].set_ylabel('Variance')
    ax[1, 1].set_title('Filter Spread of MC')
    ax[1, 1].legend()

    plt.tight_layout()
    plt.show()

interact( # Interactive Sliders
    interactive_ukf,
    r_true=FloatSlider(value=2.8, min=1, max=4, step=0.01, description='True r'),
    Q=FloatSlider(value=1e-5, min=1e-15, max=1e-2, step=1e-6, readout_format='.1e', description='Q'),
    n_steps=IntSlider(value=100, min=100, max=500, step=50, description='Iterations'),
    alpha=FloatSlider(value=1e-3, min=1e-4, max=2e-1, step=1e-4, readout_format='.1e', description='Alpha'),
    beta=FloatSlider(value=2, min=0, max=4, step=0.1, description='Beta'),
    kappa=FloatSlider(value=1, min=-1, max=3, step=0.1, description='Kappa'))

interactive(children=(FloatSlider(value=2.8, description='True r', max=4.0, min=1.0, step=0.01), FloatSlider(v…

<function __main__.interactive_ukf(r_true=2.8, Q=1e-05, n_steps=100, alpha=0.001, beta=2, kappa=0)>